In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
import os
import re
from rasterio.mask import mask as rst_mask
from collections import defaultdict
from shapely.geometry import mapping
import rioxarray

In [2]:
def find_hourly_ffp_shapefiles(target_date, file_directory):
    
    target_date = f"{target_date[:4]}-{target_date[4:6]}-{target_date[6:]}"
    pattern = re.compile(rf'_{target_date}_\d{{1,2}}\.')
    

    # List to store matching files
    matching_files = []

    # Loop through all files in the directory
    for filename in os.listdir(file_directory):
        if filename.endswith('.shp'):  # Only check .xyz files
            # Search for the pattern in the filename
            if pattern.search(filename):
                matching_files.append(filename)

    return matching_files

In [3]:
def extract_ffp_ndvi_to_raster(PS_image_dir, hr_ffp_dir, out_dir):

    # Hourly FFP directory & PS dir & Imputed PS_NDVI directory
    Hourly_FFP_dir = hr_ffp_dir #UC1_hourly_ffp_dir
    PS_files_dir = PS_image_dir
    output_dir_name =  out_dir # ='Hourly_FFP_NDVI_UC1'

    PS_ndvi_dir = os.path.join(PS_files_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed')
    output_directory = os.path.join(PS_ndvi_dir, output_dir_name)
    os.makedirs(output_directory, exist_ok=True)

    for ndvi_file in os.listdir(PS_ndvi_dir):
        ndvi_file_path = os.path.join(PS_ndvi_dir, ndvi_file)
        ndvi_date = ndvi_file[:8]
        ndvi_date_shapefile = find_hourly_ffp_shapefiles(ndvi_date, Hourly_FFP_dir)
        if not ndvi_date_shapefile:
            continue
            
        for h_shpfile in ndvi_date_shapefile:
            ffp_hour = h_shpfile[33:-4]
            # Read the shapefile using GeoPandas
            h_shpfile_path = os.path.join(Hourly_FFP_dir, h_shpfile)
            shapefile = gpd.read_file(h_shpfile_path)
        
            # Ensure the geometries are valid (in case there are any issues with the shapes)
            shapefile = shapefile[shapefile.is_valid]

            try:
                # Open the raster file
                with rasterio.open(ndvi_file_path) as src:

                    if shapefile.crs != src.crs:
                        shapefile = shapefile.to_crs(src.crs)

                    # Convert the geometry to GeoJSON format for use in rasterio
                    geometries = [mapping(geometry) for geometry in shapefile.geometry]
                    
                    # Clip the raster using the shapefile geometry
                    try:
                        out_image, out_transform = rst_mask(src, geometries, crop=True)
                    
                        # Update the metadata to match the new clipped raster
                        out_meta = src.meta.copy()
                        out_meta.update({
                            "driver": "GTiff",
                            "height": out_image.shape[1],
                            "width": out_image.shape[2],
                            "transform": out_transform
                        })
                    
                        # Write the clipped raster to the output file
                        output_raster_path = os.path.join(output_directory, ndvi_file.replace('.tif', f'_{ffp_hour}_HFFP.tif'))
                        with rasterio.open(output_raster_path, "w", **out_meta) as dest:
                            dest.write(out_image)
                    except:
                        print(f'Skipping {ndvi_date}_{ffp_hour} due to ffp error')
                        
            except ValueError as e:
                if str(e) == 'Input shapes do not overlap raster.':
                    print(f"Warning: The shapefile {h_shpfile} does not overlap the raster.")
                else:
                    raise

#### Generate ffp ndvi raster

In [4]:
# PA
# Gatesburg_2019_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2019')
# Gatesburg_2020_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2020')
# Gatesburg_2021_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2021')
# Gatesburg_2022_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2022')
# Gatesburg_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2023')
# Gatesburg_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2024')
# US_HWB_2017_PS_dir = os.path.join(os.getcwd(), 'Data', 'US-HWB')

# CA 
# Bi1_2018_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2018')
#Bi1_2019_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2019')
# Bi1_2020_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2020')
# Bi1_2021_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2021')
# Bi1_2022_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2022')
# Bi1_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2023')
# Bi1_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2024')

# Bi2_2018_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2018')
# Bi2_2019_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2019')
# Bi2_2020_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2020')
# Bi2_2021_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2021')
# Bi2_2022_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2022')
# Bi2_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2023')
# Bi2_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2024')

# Tw3_2017_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Tw3', '2017')
# Tw3_2018_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Tw3', '2018')

# IL (yet to be downloaded and checked)
UiABC_2018_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2018')
UiABC_2019_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2019')
UiABC_2020_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2020')
UiABC_2021_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2021')
UiABC_2022_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2022')
UiABC_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2023')
UiABC_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2024')

# IN (yet to be downloaded and checked)
VT12_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'IN', 'US-VT12', '2023')
VT12_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'IN', 'US-VT12', '2024')

In [ ]:
import time, os

while not os.path.exists(os.path.join(os.path.dirname(os.getcwd()), '2_EC_footprint_area', 'FFP_run_flag')):
    print("Waiting for Notebook FFP to finish...")
    time.sleep(3600)  # check every 60 seconds

print('run')

Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...
Waiting for Notebook FFP to finish...


In [5]:
### PA
# hourly_ffp_dir = os.path.join(os.path.dirname(os.getcwd()), '2_EC_footprint_area', 'US-UC1_footprints_hourly')
# extract_ffp_ndvi_to_raster(Gatesburg_2023_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UC1')

# hourly_ffp_dir = os.path.join(os.path.dirname(os.getcwd()), '2_EC_footprint_area', 'US-UC2_footprints_hourly')
# extract_ffp_ndvi_to_raster(Gatesburg_2023_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UC2')

### CA
# hourly_ffp_dir = os.path.join(os.path.dirname(os.getcwd()), '2_EC_footprint_area', 'US-Bi1_footprints_hourly')
# extract_ffp_ndvi_to_raster(Bi1_2018_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi1')
#extract_ffp_ndvi_to_raster(Bi1_2019_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi1')
# extract_ffp_ndvi_to_raster(Bi1_2020_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi1')
# extract_ffp_ndvi_to_raster(Bi1_2021_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi1')
# extract_ffp_ndvi_to_raster(Bi1_2022_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi1')
# extract_ffp_ndvi_to_raster(Bi1_2023_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi1')
# extract_ffp_ndvi_to_raster(Bi1_2024_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi1')

# hourly_ffp_dir = os.path.join(os.path.dirname(os.getcwd()), '2_EC_footprint_area', 'US-Bi2_footprints_hourly')
# extract_ffp_ndvi_to_raster(Bi2_2018_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi2')
# extract_ffp_ndvi_to_raster(Bi2_2019_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi2')
# extract_ffp_ndvi_to_raster(Bi2_2020_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi2')
# extract_ffp_ndvi_to_raster(Bi2_2021_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi2')
# extract_ffp_ndvi_to_raster(Bi2_2022_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi2')
# extract_ffp_ndvi_to_raster(Bi2_2023_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi2')
# extract_ffp_ndvi_to_raster(Bi2_2024_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Bi2')

# hourly_ffp_dir = os.path.join(os.path.dirname(os.getcwd()), '2_EC_footprint_area', 'US-Tw3_footprints_hourly')
# extract_ffp_ndvi_to_raster(Tw3_2017_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Tw3')
# extract_ffp_ndvi_to_raster(Tw3_2018_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_Tw3')

### IL (yet to be downloaded and checked)
hourly_ffp_dir = os.path.join(os.path.dirname(os.getcwd()), '2_EC_footprint_area', 'US-UiA_footprints_hourly')
extract_ffp_ndvi_to_raster(UiABC_2018_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiA')
extract_ffp_ndvi_to_raster(UiABC_2019_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiA')
extract_ffp_ndvi_to_raster(UiABC_2020_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiA')
extract_ffp_ndvi_to_raster(UiABC_2021_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiA')
extract_ffp_ndvi_to_raster(UiABC_2022_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiA')
extract_ffp_ndvi_to_raster(UiABC_2023_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiA')
extract_ffp_ndvi_to_raster(UiABC_2024_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiA')

hourly_ffp_dir = os.path.join(os.path.dirname(os.getcwd()), '2_EC_footprint_area', 'US-UiB_footprints_hourly')
extract_ffp_ndvi_to_raster(UiABC_2018_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiB')
extract_ffp_ndvi_to_raster(UiABC_2019_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiB')
extract_ffp_ndvi_to_raster(UiABC_2020_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiB')
extract_ffp_ndvi_to_raster(UiABC_2021_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiB')
extract_ffp_ndvi_to_raster(UiABC_2022_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiB')
extract_ffp_ndvi_to_raster(UiABC_2023_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiB')
extract_ffp_ndvi_to_raster(UiABC_2024_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiB')

hourly_ffp_dir = os.path.join(os.path.dirname(os.getcwd()), '2_EC_footprint_area', 'US-UiC_footprints_hourly')
extract_ffp_ndvi_to_raster(UiABC_2018_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiC')
extract_ffp_ndvi_to_raster(UiABC_2019_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiC')
extract_ffp_ndvi_to_raster(UiABC_2020_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiC')
extract_ffp_ndvi_to_raster(UiABC_2021_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiC')
extract_ffp_ndvi_to_raster(UiABC_2022_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiC')
extract_ffp_ndvi_to_raster(UiABC_2023_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiC')
extract_ffp_ndvi_to_raster(UiABC_2024_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_UiC')

### IN (yet to be downloaded and checked)
hourly_ffp_dir = os.path.join(os.path.dirname(os.getcwd()), '2_EC_footprint_area', 'US-VT1_footprints_hourly')
extract_ffp_ndvi_to_raster(VT12_2023_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_VT1')
extract_ffp_ndvi_to_raster(VT12_2024_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_VT1')

hourly_ffp_dir = os.path.join(os.path.dirname(os.getcwd()), '2_EC_footprint_area', 'US-VT2_footprints_hourly')
extract_ffp_ndvi_to_raster(VT12_2023_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_VT2')
extract_ffp_ndvi_to_raster(VT12_2024_PS_dir, hourly_ffp_dir, 'Hourly_FFP_NDVI_VT2')

Skipping 20180526_10 due to ffp error
Skipping 20180526_11 due to ffp error
Skipping 20180526_12 due to ffp error
Skipping 20180526_13 due to ffp error
Skipping 20180526_14 due to ffp error
Skipping 20180526_15 due to ffp error
Skipping 20180526_16 due to ffp error
Skipping 20180526_17 due to ffp error
Skipping 20180526_18 due to ffp error
Skipping 20180526_6 due to ffp error
Skipping 20180526_7 due to ffp error
Skipping 20180526_8 due to ffp error
Skipping 20180526_9 due to ffp error
Skipping 20180527_10 due to ffp error
Skipping 20180527_11 due to ffp error
Skipping 20180527_12 due to ffp error
Skipping 20180527_13 due to ffp error
Skipping 20180527_14 due to ffp error
Skipping 20180527_15 due to ffp error
Skipping 20180527_16 due to ffp error
Skipping 20180527_17 due to ffp error
Skipping 20180527_18 due to ffp error
Skipping 20180527_6 due to ffp error
Skipping 20180527_7 due to ffp error
Skipping 20180527_8 due to ffp error
Skipping 20180527_9 due to ffp error
Skipping 20180528_10

#### Extract ffp mean NDVI

In [6]:
def ffp_NDVI_raster_to_gdf_with_datetime(multiband_raster, Band_name='NDVI'):
    
    image_date = os.path.basename(multiband_raster)[:8]   # NOTE: it is important for all the ffp ndvi files to have the same format (hard-coded here)
    image_hour = os.path.basename(multiband_raster)[25:-9]
    
    with rioxarray.open_rasterio(multiband_raster) as raster:
        num_bands = raster.rio.count
        for i in range(1, num_bands+1):
            
            selected_data = raster.isel(band=i-1)
            raster.name = Band_name
            nodata_value = raster.rio.nodata
            #print(nodata_value)
            df = raster.squeeze().to_dataframe().reset_index()
            geometry = gpd.points_from_xy(df.x, df.y)
            gdf = gpd.GeoDataFrame(df, crs=raster.rio.crs, geometry=geometry).to_crs(epsg=4326)
            gdf['latitude'] = gdf.geometry.y
            gdf['longitude'] = gdf.geometry.x

            gdf = gdf.drop(columns=['x', 'y', 'spatial_ref', 'band'], errors='ignore')
            gdf['Date'] = image_date
            gdf['hour'] = image_hour
            
            return gdf[['Date', 'hour', 'latitude', 'longitude', 'NDVI', 'geometry']]

In [7]:
# Hourly FFP NDVI directory
### PA
# UC1_ffp_NDVI_2023_dir = os.path.join(Gatesburg_2023_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UC1')
# UC2_ffp_NDVI_2023_dir = os.path.join(Gatesburg_2023_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UC2')

### CA
# Bi1_ffp_NDVI_2018_dir = os.path.join(Bi1_2018_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi1')
# Bi1_ffp_NDVI_2019_dir = os.path.join(Bi1_2019_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi1')
# Bi1_ffp_NDVI_2020_dir = os.path.join(Bi1_2020_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi1')
# Bi1_ffp_NDVI_2021_dir = os.path.join(Bi1_2021_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi1')
# Bi1_ffp_NDVI_2022_dir = os.path.join(Bi1_2022_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi1')
# Bi1_ffp_NDVI_2023_dir = os.path.join(Bi1_2023_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi1')
# Bi1_ffp_NDVI_2024_dir = os.path.join(Bi1_2024_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi1')

# Bi2_ffp_NDVI_2018_dir = os.path.join(Bi2_2018_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi2')
# Bi2_ffp_NDVI_2019_dir = os.path.join(Bi2_2019_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi2')
# Bi2_ffp_NDVI_2020_dir = os.path.join(Bi2_2020_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi2')
# Bi2_ffp_NDVI_2021_dir = os.path.join(Bi2_2021_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi2')
# Bi2_ffp_NDVI_2022_dir = os.path.join(Bi2_2022_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi2')
# Bi2_ffp_NDVI_2023_dir = os.path.join(Bi2_2023_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi2')
# Bi2_ffp_NDVI_2024_dir = os.path.join(Bi2_2024_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Bi2')

# Tw3_ffp_NDVI_2017_dir = os.path.join(Tw3_2017_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Tw3')
# Tw3_ffp_NDVI_2018_dir = os.path.join(Tw3_2018_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_Tw3')

### IL (yet to be downloaded and checked)
UiA_ffp_NDVI_2018_dir = os.path.join(UiABC_2018_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiA')
UiA_ffp_NDVI_2019_dir = os.path.join(UiABC_2019_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiA')
UiA_ffp_NDVI_2020_dir = os.path.join(UiABC_2020_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiA')
UiA_ffp_NDVI_2021_dir = os.path.join(UiABC_2021_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiA')
UiA_ffp_NDVI_2022_dir = os.path.join(UiABC_2022_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiA')
UiA_ffp_NDVI_2023_dir = os.path.join(UiABC_2023_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiA')
UiA_ffp_NDVI_2024_dir = os.path.join(UiABC_2024_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiA')

UiB_ffp_NDVI_2018_dir = os.path.join(UiABC_2018_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiB')
UiB_ffp_NDVI_2019_dir = os.path.join(UiABC_2019_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiB')
UiB_ffp_NDVI_2020_dir = os.path.join(UiABC_2020_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiB')
UiB_ffp_NDVI_2021_dir = os.path.join(UiABC_2021_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiB')
UiB_ffp_NDVI_2022_dir = os.path.join(UiABC_2022_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiB')
UiB_ffp_NDVI_2023_dir = os.path.join(UiABC_2023_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiB')
UiB_ffp_NDVI_2024_dir = os.path.join(UiABC_2024_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiB')

UiC_ffp_NDVI_2018_dir = os.path.join(UiABC_2018_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiC')
UiC_ffp_NDVI_2019_dir = os.path.join(UiABC_2019_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiC')
UiC_ffp_NDVI_2020_dir = os.path.join(UiABC_2020_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiC')
UiC_ffp_NDVI_2021_dir = os.path.join(UiABC_2021_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiC')
UiC_ffp_NDVI_2022_dir = os.path.join(UiABC_2022_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiC')
UiC_ffp_NDVI_2023_dir = os.path.join(UiABC_2023_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiC')
UiC_ffp_NDVI_2024_dir = os.path.join(UiABC_2024_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_UiC')

### IN (yet to be downloaded and checked)
VT1_ffp_NDVI_2023_dir = os.path.join(VT12_2023_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_VT1')
VT1_ffp_NDVI_2024_dir = os.path.join(VT12_2024_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_VT1')

VT2_ffp_NDVI_2023_dir = os.path.join(VT12_2023_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_VT2')
VT2_ffp_NDVI_2024_dir = os.path.join(VT12_2024_PS_dir, 'NDVI', 'standardized_crs', 'NDVI_imputed', 'Hourly_FFP_NDVI_VT2')


In [10]:
# Export hourly mean PS NDVI to csv files
### PA and CA
# hourly_ffp_ndvi_dir_list = [Bi2_ffp_NDVI_2019_dir, Bi2_ffp_NDVI_2020_dir,
#                             Bi2_ffp_NDVI_2021_dir,Bi2_ffp_NDVI_2022_dir, Bi2_ffp_NDVI_2023_dir,  Bi2_ffp_NDVI_2024_dir,

#                             Tw3_ffp_NDVI_2017_dir, Tw3_ffp_NDVI_2018_dir,
#                             ]

### IL and IN (for NOV)
hourly_ffp_ndvi_dir_list = [UiA_ffp_NDVI_2018_dir, UiA_ffp_NDVI_2019_dir, UiA_ffp_NDVI_2020_dir,
                            UiA_ffp_NDVI_2021_dir, UiA_ffp_NDVI_2022_dir, UiA_ffp_NDVI_2023_dir,  UiA_ffp_NDVI_2024_dir,

                            UiB_ffp_NDVI_2018_dir, UiB_ffp_NDVI_2019_dir, UiB_ffp_NDVI_2020_dir,
                            UiB_ffp_NDVI_2021_dir, UiB_ffp_NDVI_2022_dir, UiB_ffp_NDVI_2023_dir,  UiB_ffp_NDVI_2024_dir,

                            UiC_ffp_NDVI_2018_dir, UiC_ffp_NDVI_2019_dir, UiC_ffp_NDVI_2020_dir,
                            UiC_ffp_NDVI_2021_dir, UiC_ffp_NDVI_2022_dir, UiC_ffp_NDVI_2023_dir,  UiC_ffp_NDVI_2024_dir,

                            VT1_ffp_NDVI_2023_dir,  VT1_ffp_NDVI_2024_dir,
                            VT2_ffp_NDVI_2023_dir,  VT2_ffp_NDVI_2024_dir,
                            ]

for Hourly_ffp_NDVI_raster_dir in hourly_ffp_ndvi_dir_list:
    try:
        all_gdfs = []
        for filename in os.listdir(Hourly_ffp_NDVI_raster_dir):
            # print(filename)
            if filename.endswith('.tif'):  # Assuming all rasters are in .tif format
                raster_path = os.path.join(Hourly_ffp_NDVI_raster_dir, filename)
                all_gdfs.append(ffp_NDVI_raster_to_gdf_with_datetime(raster_path))

        combined_gdf = gpd.GeoDataFrame(pd.concat(all_gdfs, ignore_index=True))
        filtered_combined_gdf = combined_gdf[combined_gdf['NDVI']!=0]

        # Export only mean value of ndvi
        path_basename = os.path.basename(Hourly_ffp_NDVI_raster_dir)
        year = re.search(r'\b(19|20)\d{2}\b', Hourly_ffp_NDVI_raster_dir).group()
        filtered_combined_gdf.groupby(['Date', 'hour'])['NDVI'].mean().reset_index().to_csv(f'PS_AVE_{path_basename}_{year}.csv', index=False)
    except:
        print(f'Error for {os.path.basename(Hourly_ffp_NDVI_raster_dir)}')

Error for Hourly_FFP_NDVI_UiA
Error for Hourly_FFP_NDVI_UiA
Error for Hourly_FFP_NDVI_UiA
Error for Hourly_FFP_NDVI_UiA
Error for Hourly_FFP_NDVI_UiA
Error for Hourly_FFP_NDVI_UiA
20240528_PS_NDVI_imputed_11_HFFP.tif
20240528_PS_NDVI_imputed_12_HFFP.tif
20240528_PS_NDVI_imputed_13_HFFP.tif
20240528_PS_NDVI_imputed_14_HFFP.tif
20240528_PS_NDVI_imputed_15_HFFP.tif
20240528_PS_NDVI_imputed_16_HFFP.tif
20240528_PS_NDVI_imputed_17_HFFP.tif
20240528_PS_NDVI_imputed_18_HFFP.tif
20240529_PS_NDVI_imputed_10_HFFP.tif
20240529_PS_NDVI_imputed_11_HFFP.tif
20240529_PS_NDVI_imputed_12_HFFP.tif
20240529_PS_NDVI_imputed_13_HFFP.tif
20240529_PS_NDVI_imputed_14_HFFP.tif
20240529_PS_NDVI_imputed_15_HFFP.tif
20240529_PS_NDVI_imputed_16_HFFP.tif
20240529_PS_NDVI_imputed_17_HFFP.tif
20240529_PS_NDVI_imputed_6_HFFP.tif
20240529_PS_NDVI_imputed_7_HFFP.tif
20240529_PS_NDVI_imputed_8_HFFP.tif
20240529_PS_NDVI_imputed_9_HFFP.tif
20240530_PS_NDVI_imputed_10_HFFP.tif
20240530_PS_NDVI_imputed_11_HFFP.tif
20240530_P

In [ ]:
path = os.path.join(os.getcwd(), 'FFP_NDVI_run_flag')

os.makedirs(path, exist_ok=True)